# Image Classification - Exploratory Notebook

This notebook provides an interactive environment for exploring the image classification project.

## 1. Setup and Imports

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# Add src to path
sys.path.append('../src')

from config import *
from dataset import load_data_from_folders, get_transforms, ImageClassificationDataset
from model import get_model
from utils import get_detailed_metrics, plot_confusion_matrix

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## 2. Load and Explore Data

In [ ]:
# Set your data directory path
data_dir = RAW_DATA_DIR  # or specify your own path

# Check if data exists
if os.path.exists(data_dir):
    print(f'Data directory: {data_dir}')
    print(f'Contents: {os.listdir(data_dir)}')
else:
    print(f'Data directory does not exist: {data_dir}')
    print('Please organize your data in: data_dir/class_name/image.jpg structure')

In [ ]:
# Load data (if available)
if os.path.exists(data_dir) and len(os.listdir(data_dir)) > 0:
    train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names = \
        load_data_from_folders(data_dir, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT, RANDOM_SEED)
    
    print(f'Number of classes: {len(class_names)}')
    print(f'Classes: {class_names}')
    print(f'\nDataset splits:')
    print(f'  Train: {len(train_paths)} samples')
    print(f'  Val: {len(val_paths)} samples')
    print(f'  Test: {len(test_paths)} samples')
    print(f'  Total: {len(train_paths) + len(val_paths) + len(test_paths)} samples')

## 3. Visualize Sample Images

In [ ]:
# Visualize random samples from each class
if 'class_names' in locals():
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for i in range(min(10, len(train_paths))):
        img = Image.open(train_paths[i])
        axes[i].imshow(img)
        axes[i].set_title(f'{class_names[train_labels[i]]}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

## 4. Analyze Class Distribution

In [ ]:
# Plot class distribution
if 'train_labels' in locals():
    from collections import Counter
    
    train_counter = Counter(train_labels)
    val_counter = Counter(val_labels)
    test_counter = Counter(test_labels)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))
    
    # Train distribution
    ax1.bar([class_names[i] for i in sorted(train_counter.keys())], 
            [train_counter[i] for i in sorted(train_counter.keys())])
    ax1.set_title('Training Set Distribution')
    ax1.set_xlabel('Class')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=45)
    
    # Validation distribution
    ax2.bar([class_names[i] for i in sorted(val_counter.keys())], 
            [val_counter[i] for i in sorted(val_counter.keys())])
    ax2.set_title('Validation Set Distribution')
    ax2.set_xlabel('Class')
    ax2.set_ylabel('Count')
    ax2.tick_params(axis='x', rotation=45)
    
    # Test distribution
    ax3.bar([class_names[i] for i in sorted(test_counter.keys())], 
            [test_counter[i] for i in sorted(test_counter.keys())])
    ax3.set_title('Test Set Distribution')
    ax3.set_xlabel('Class')
    ax3.set_ylabel('Count')
    ax3.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 5. Test Data Augmentation

In [ ]:
# Visualize augmented images
if 'train_paths' in locals() and len(train_paths) > 0:
    sample_image = Image.open(train_paths[0]).convert('RGB')
    sample_array = np.array(sample_image)
    
    transform = get_transforms(IMG_HEIGHT, IMG_WIDTH, augment=True)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i in range(8):
        augmented = transform(image=sample_array)
        img = augmented['image'].permute(1, 2, 0).numpy()
        # Denormalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].set_title(f'Augmented {i+1}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

## 6. Model Architecture

In [ ]:
# Create and inspect model
if 'class_names' in locals():
    num_classes = len(class_names)
else:
    num_classes = NUM_CLASSES

model = get_model(MODEL_NAME, num_classes, PRETRAINED, FREEZE_BACKBONE)

print(f'Model: {MODEL_NAME}')
print(f'Number of classes: {num_classes}')
print(f'Pretrained: {PRETRAINED}')
print(f'\nModel Summary:')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Non-trainable parameters: {total_params - trainable_params:,}')

## 7. Load and Evaluate Trained Model

In [ ]:
# Load trained model (if available)
model_path = os.path.join(SAVED_MODELS_DIR, 'best_model.pth')

if os.path.exists(model_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f'Model loaded from {model_path}')
    print(f'Training epoch: {checkpoint.get("epoch", "unknown")}')
    print(f'Validation accuracy: {checkpoint.get("val_acc", "unknown"):.4f}')
else:
    print(f'No trained model found at {model_path}')
    print('Train a model first using train.py')

## 8. Make Predictions

In [ ]:
# Make predictions on sample images
if os.path.exists(model_path) and 'test_paths' in locals() and len(test_paths) > 0:
    import torch.nn.functional as F
    
    transform = get_transforms(IMG_HEIGHT, IMG_WIDTH, augment=False)
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for i in range(min(10, len(test_paths))):
        # Load and preprocess image
        img = Image.open(test_paths[i]).convert('RGB')
        img_array = np.array(img)
        transformed = transform(image=img_array)
        img_tensor = transformed['image'].unsqueeze(0).to(device)
        
        # Predict
        with torch.no_grad():
            output = model(img_tensor)
            probs = F.softmax(output, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred_class].item()
        
        # Plot
        axes[i].imshow(img)
        true_label = class_names[test_labels[i]]
        pred_label = class_names[pred_class]
        color = 'green' if pred_class == test_labels[i] else 'red'
        axes[i].set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.2f})', 
                         color=color)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

## 9. Training History Visualization

In [ ]:
# Visualize training history (if available)
history_plot = os.path.join(SAVED_MODELS_DIR, 'training_history.png')
confusion_matrix_plot = os.path.join(SAVED_MODELS_DIR, 'confusion_matrix.png')

if os.path.exists(history_plot):
    img = Image.open(history_plot)
    plt.figure(figsize=(15, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training History')
    plt.show()

if os.path.exists(confusion_matrix_plot):
    img = Image.open(confusion_matrix_plot)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()

## 10. Next Steps

1. **Prepare your data**: Organize images in the structure `data/raw/class_name/image.jpg`
2. **Train the model**: Run `python train.py --data_dir data/raw`
3. **Make predictions**: Run `python predict.py --model_path models/saved_models/best_model.pth --image_path path/to/image.jpg`
4. **Experiment**: Modify hyperparameters in `src/config.py` and retrain